## The whole thing

#### Idea :

For each formation day D:
- Take top K pairs for day D
- Set trading day = next day D+1

For each pair:
- decide leader/follower + lag L
- compute leader Bollinger signal on day D+1
- shift signal by L minutes
- compute pnl on follower returns
- Average pnl across pairs → portfolio pnl series for that day
- Sum over minutes → one daily return

Repeat for all days → equity curve + metrics

In [17]:
def backtest_ocp_strategy(
    ret: pd.DataFrame,
    pairs_df: pd.DataFrame,
    K=10,
    d=20,
    k=2.0,
    r_cost=0.0,
    cost_bps=1.0,
):
    """
    ret: minute returns, index=datetime, columns=tickers
    pairs_df: formation_date, a, b, l_hat, sigma_l, rank
    Output: DataFrame of daily strategy returns + equity curve
    """

    # to make sure we have a date column derived from timestamps
    #since in our data there are timestamps (no explicit column for dates)
    ret = ret.sort_index()
    ret_dates = pd.Index(ret.index.date).unique().tolist()
    ret_dates = sorted(ret_dates)

    daily_results = []

    # group pairs by formation_date
    for formation_day, g in pairs_df.groupby("formation_date"):
        formation_day = pd.to_datetime(formation_day).date()

        # trading day = next day in your returns data
        if formation_day not in ret_dates:
            continue
        trade_day = get_next_trading_day(ret_dates, formation_day)
        if trade_day is None:
            continue

        # minute returns for trading day only
        day_mask = (ret.index.date == trade_day)
        Rday = ret.loc[day_mask]
        if Rday.empty:
            continue

        # top K pairs
        g = g.sort_values("rank").head(K)

        pair_pnls = []

        for _, row in g.iterrows():
            a = row["a"]
            b = row["b"]
            l_hat = int(row["l_hat"])

            leader, follower, L = leader_follower(a, b, l_hat)
            if leader is None:
                continue
            if leader not in Rday.columns or follower not in Rday.columns:
                continue

            r_leader = Rday[leader].dropna()
            r_follower = Rday[follower].dropna()

            # align to the same timestamps (important!)
            common_idx = r_leader.index.intersection(r_follower.index)
            r_leader = r_leader.loc[common_idx]
            r_follower = r_follower.loc[common_idx]

            # raw signal based on leader
            raw_sig = bollinger_raw_signal(r_leader, d=d, k=k, r_cost=r_cost)

            # EXECUTION: shift by L minutes
            # Because your index is minute timestamps, shifting by periods is correct
            exec_sig = raw_sig.shift(L).reindex(common_idx).fillna(0).astype(np.int8)

            # Option A: position = signal each minute (1-minute holding)
            pos = exec_sig

            pnl_pair = pnl_from_positions(pos, r_follower, cost_bps=cost_bps)
            pair_pnls.append(pnl_pair)

        if not pair_pnls:
            continue

        # portfolio = average across pairs (equal weight)
        pnl_port = pd.concat(pair_pnls, axis=1).mean(axis=1)

        # store results
        daily_return = pnl_port.sum()
        daily_results.append(
            {"date": trade_day, "daily_return": daily_return}
        )

    out = pd.DataFrame(daily_results).set_index("date").sort_index()
    out["equity"] = (1 + out["daily_return"]).cumprod()
    return out

## Making things more compatible with previous codes:

In [26]:
"""
This script assumes we already have:
  1) Preprocessed per-ticker parquet files in data_dir, each with columns:
        - "timestamp" (minute timestamps)
        - "return"    (minute returns)
     and filenames like: {TICKER}.parquet

  2) ocp.py functions available (importable), specifically:
        - build_daily_return_matrices(...)
        - build_daily_pairs(...)
        - ocp_run_all_days_fast(...)

What we do:
  - Run OCP over all formation days (or load saved OCP results)
  - Trade the "top_k" pairs on the NEXT trading day
  - Compute daily strategy returns + equity curve + basic metrics
  - Save outputs (CSV) and plot
"""

from __future__ import annotations

import numpy as np
import pandas as pd
from pathlib import Path
from typing import Dict, List, Optional, Tuple

In [30]:
#1. helper utils functions : check
def ensure_date(x) -> "datetime.date":
    """Convert x to a python date (no time)."""
    return pd.to_datetime(x).date()


def get_sorted_days(daily_matrices: Dict) -> List:
    """Return sorted list of day keys (python date)."""
    return sorted(daily_matrices.keys())


def next_trading_day(sorted_days: List, day) -> Optional:
    """Return next element in sorted_days after day, or None."""
    try:
        i = sorted_days.index(day)
    except ValueError:
        return None
    return sorted_days[i + 1] if i + 1 < len(sorted_days) else None

In [ ]:
#2. Signals and PnL : check


#helps decide when to trade
def bollinger_signal(
    r: pd.Series,
    window: int = 20,
    k: float = 2.0,
    r_cost: float = 0.0
) -> pd.Series:
    """
    Build raw signal from leader returns using Bollinger bands.
    Output values in {-1, 0, +1} aligned with r's index.

    r_cost : transaction cost
    If r_cost > 0: enforce magnitude threshold
      long if r > upper and r > r_cost
      short if r < lower and r < -r_cost
    """

    #At each minute, we look at the last "window" minutes and we compute
    mu = r.rolling(window, min_periods=window).mean()
    sd = r.rolling(window, min_periods=window).std(ddof=0)

    #rule to detect abnormal moves.
    #upper = very unusually high
    #lower = very unusually low
    upper = mu + k * sd
    lower = mu - k * sd

    sig = pd.Series(0, index=r.index, dtype=np.int8)
    sig[(r > upper) & (r > r_cost)] = 1
    sig[(r < lower) & (r < -r_cost)] = -1
    return sig

#Compute how much win or lose
def pnl_from_positions(
    pos: pd.Series,
    r_follower: pd.Series,
    cost_bps: float = 0.0
) -> pd.Series:
    """
    Compute per-minute PnL series:
        pnl[t] = pos[t] * r_follower[t] - costs_on_changes

    cost_bps applies when position changes:
        turnover = |pos[t] - pos[t-1]|
        cost = turnover * (cost_bps * 1e-4) / 2
    """

    #to align indexes
    pos = pos.reindex(r_follower.index).fillna(0).astype(float)
    r_follower = r_follower.fillna(0.0)

    pnl = pos * r_follower

    #optional transaction cost
    if cost_bps and cost_bps > 0:
        turnover = pos.diff().abs().fillna(0.0)  # 0->1:1, -1->+1:2
        pnl -= turnover * (cost_bps * 1e-4) / 2.0

    return pnl


In [ ]:
# 3. Trade one day (top pairs)
def trade_one_day(
    day_matrix: pd.DataFrame,     # minutes x tickers returns for the trading day
    top_pairs_df: pd.DataFrame,   # columns: leader, follower, l_hat (float)
    window: int = 20,
    k: float = 2.0,
    r_cost: float = 0.0,
    cost_bps: float = 0.0,
) -> pd.Series:
    """
    For each pair:
      - compute leader signal on this trading day
      - shift signal by L = round(abs(l_hat)) minutes
      - apply to follower returns
    Aggregate equal-weight across pairs.
    Return: portfolio pnl series (minute indexed).
    """
    pair_pnls = []

    for _, row in top_pairs_df.iterrows():
        leader = row["leader"]
        follower = row["follower"]
        L = int(round(abs(float(row["l_hat"]))))

        if L <= 0:
            continue
        if leader not in day_matrix.columns or follower not in day_matrix.columns:
            continue

        rL = day_matrix[leader]
        rF = day_matrix[follower]

        # Align timestamps (usually already aligned, but safe)
        idx = rL.index.intersection(rF.index)
        rL = rL.loc[idx]
        rF = rF.loc[idx]

        raw_sig = bollinger_signal(rL, window=window, k=k, r_cost=r_cost)

        # Execute on follower after lag L (shift forward by L minutes)
        pos = raw_sig.shift(L).reindex(idx).fillna(0).astype(np.int8)

        pnl_pair = pnl_from_positions(pos, rF, cost_bps=cost_bps)
        pair_pnls.append(pnl_pair)

    if not pair_pnls:
        return pd.Series(0.0, index=day_matrix.index)

    pnl_port = pd.concat(pair_pnls, axis=1).mean(axis=1)  # equal weight
    return pnl_port

In [ ]:
# 4. Backtest across all days
def backtest_from_ocp_results(
    daily_matrices: Dict,
    ocp_results: pd.DataFrame,    # columns include: date, leader, follower, l_hat
    top_k: int = 10,
    window: int = 20,
    k: float = 2.0,
    r_cost: float = 0.0,
    cost_bps: float = 0.0,
) -> pd.DataFrame:
    """
    Formation day = ocp_results['date']
    Trading day   = next trading day in daily_matrices

    Returns DataFrame indexed by trade date with:
      - strategy_return (sum of minute pnl)
      - equity (cumprod of 1+daily return)
    """
    days = get_sorted_days(daily_matrices)

    df = ocp_results.copy()
    df["date"] = pd.to_datetime(df["date"]).dt.date

    rows = []

    for formation_day, g in df.groupby("date"):
        if formation_day not in daily_matrices:
            continue

        trade_day = next_trading_day(days, formation_day)
        if trade_day is None:
            continue

        # top_k pairs for this formation day
        # ocp_run_all_days_fast already returns top_k per day, but we keep it safe:
        top_pairs = g.head(top_k)

        day_matrix = daily_matrices[trade_day]
        pnl_minute = trade_one_day(
            day_matrix=day_matrix,
            top_pairs_df=top_pairs,
            window=window,
            k=k,
            r_cost=r_cost,
            cost_bps=cost_bps,
        )

        daily_ret = float(pnl_minute.sum())
        rows.append({"date": trade_day, "strategy_return": daily_ret})

    out = pd.DataFrame(rows).set_index("date").sort_index()
    out["equity"] = (1.0 + out["strategy_return"]).cumprod()
    return out

In [ ]:
#5. Benchmark + metrics

def make_equal_weight_benchmark(daily_matrices: Dict) -> pd.Series:
    """
    Creates a simple daily benchmark using equal-weight average return
    across all available tickers each day.
    """
    days = sorted(daily_matrices.keys())
    bench = []
    for day in days:
        R = daily_matrices[day]  # minutes x tickers
        # equal-weight "market proxy" minute pnl:
        pnl_min = R.mean(axis=1).fillna(0.0)
        bench.append((day, float(pnl_min.sum())))
    return pd.Series(dict(bench)).sort_index()


def sharpe_ratio(daily_returns: pd.Series, annualization: int = 252) -> float:
    r = daily_returns.dropna()
    if len(r) < 2:
        return np.nan
    sd = r.std(ddof=0)
    if sd == 0:
        return np.nan
    return float(np.sqrt(annualization) * r.mean() / sd)


def max_drawdown(equity: pd.Series) -> float:
    peak = equity.cummax()
    dd = equity / peak - 1.0
    return float(dd.min())


def summarize_performance(strategy_daily: pd.Series, benchmark_daily: Optional[pd.Series] = None) -> pd.DataFrame:
    """
    Returns a small table of metrics.
    """
    res = {
        "mean_daily": float(strategy_daily.mean()),
        "std_daily": float(strategy_daily.std(ddof=0)),
        "sharpe": sharpe_ratio(strategy_daily),
        "max_drawdown": max_drawdown((1 + strategy_daily).cumprod()),
        "n_days": int(strategy_daily.dropna().shape[0]),
    }

    if benchmark_daily is not None:
        bench = benchmark_daily.reindex(strategy_daily.index).dropna()
        strat = strategy_daily.reindex(bench.index)
        res["mean_daily_bench"] = float(bench.mean())
        res["sharpe_bench"] = sharpe_ratio(bench)
        # simple outperformance:
        res["mean_daily_diff"] = float((strat - bench).mean())

    return pd.DataFrame([res])

In [ ]:
#6. Full pipeline
def run_full_pipeline(
    data_dir: Path,
    ticker_list: List[str],
    *,
    # OCP settings (match teammate defaults)
    keep_top_pairs: int = 400,
    top_k: int = 10,
    max_lag: int = 30,
    band: int = 10,
    first_n_days: Optional[int] = None,
    # Trading settings (your knobs)
    window: int = 20,
    k: float = 2.0,
    r_cost: float = 0.0,
    cost_bps: float = 0.0,
    # optional caching
    ocp_results_path: Optional[Path] = None,
) -> Tuple[pd.DataFrame, pd.DataFrame, pd.Series]:
    """
    Returns:
      results_df: index=trade day, columns=strategy_return,equity
      metrics_df: one-row table of metrics
      benchmark_daily: series of daily benchmark returns
    """
    # --- import teammate OCP functions here ---
    # Adjust module name if needed.
    from ocp import (
        build_daily_return_matrices,
        build_daily_pairs,
        ocp_run_all_days_fast,
    )

    # 1) Build daily matrices (minutes x tickers returns)
    daily_matrices = build_daily_return_matrices(
        data_dir=data_dir,
        ticker_list=ticker_list,
        timestamp_col="timestamp",
        return_col="return",
    )

    # 2) Build daily pairs list
    daily_pairs = build_daily_pairs(daily_matrices)

    # 3) Get OCP results (compute or load)
    if ocp_results_path is not None and ocp_results_path.exists():
        ocp_results = pd.read_parquet(ocp_results_path) if ocp_results_path.suffix == ".parquet" else pd.read_csv(ocp_results_path)
    else:
        ocp_results = ocp_run_all_days_fast(
            daily_matrices=daily_matrices,
            daily_pairs=daily_pairs,
            keep_top_pairs=keep_top_pairs,
            top_k=top_k,
            max_lag=max_lag,
            band=band,
            first_n_days=first_n_days,
        )
        if ocp_results_path is not None:
            if ocp_results_path.suffix == ".parquet":
                ocp_results.to_parquet(ocp_results_path, index=False)
            else:
                ocp_results.to_csv(ocp_results_path, index=False)

    # 4) Backtest strategy using OCP results
    results_df = backtest_from_ocp_results(
        daily_matrices=daily_matrices,
        ocp_results=ocp_results,
        top_k=top_k,
        window=window,
        k=k,
        r_cost=r_cost,
        cost_bps=cost_bps,
    )

    # 5) Benchmark (equal-weight proxy) + metrics
    benchmark_daily = make_equal_weight_benchmark(daily_matrices)
    benchmark_daily = benchmark_daily.reindex(results_df.index)

    metrics_df = summarize_performance(
        strategy_daily=results_df["strategy_return"],
        benchmark_daily=benchmark_daily,
    )

    return results_df, metrics_df, benchmark_daily

In [ ]:
# 7. Plotting

def plot_equity_curves(results_df: pd.DataFrame, benchmark_daily: Optional[pd.Series] = None):
    import matplotlib.pyplot as plt

    eq_strat = results_df["equity"]
    plt.figure()
    plt.plot(eq_strat.index, eq_strat.values, label="Strategy")

    if benchmark_daily is not None:
        eq_bench = (1 + benchmark_daily.fillna(0.0)).cumprod()
        plt.plot(eq_bench.index, eq_bench.values, label="Benchmark")

    plt.legend()
    plt.title("Equity Curve")
    plt.xlabel("Date")
    plt.ylabel("Equity (start=1)")
    plt.show()

In [ ]:
if __name__ == "__main__":
    # Example usage (edit!)
    data_dir = Path("PATH/TO/CLEAN_TICKER_PARQUETS")
    ticker_list = [
        # "AAPL", "MSFT", ...
    ]

    results_df, metrics_df, benchmark_daily = run_full_pipeline(
        data_dir=data_dir,
        ticker_list=ticker_list,
        ocp_results_path=Path("ocp_results.parquet"),  # cache
        first_n_days=None,  # e.g. 50 for quick test
        window=20,
        k=2.0,
        r_cost=0.0,
        cost_bps=1.0,
    )

    print(metrics_df)
    plot_equity_curves(results_df, benchmark_daily)
    results_df.to_csv("strategy_results.csv")